In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image 
from collections import OrderedDict 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, models
from torchvision.transforms import v2
import random
import seaborn as sns
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             cohen_kappa_score, matthews_corrcoef,
                             confusion_matrix, classification_report, roc_curve, auc, roc_auc_score)
from sklearn.preprocessing import label_binarize
from collections import Counter, defaultdict
from skimage.feature import hog, graycomatrix, graycoprops
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy import stats
from sklearn.model_selection import train_test_split
import timm

d:\Practise\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_dir = Path(r"D:\GG\Datasets\Eye Disease Image Dataset\Fourier_Enhanced_Original_Dataset")
image_height = 224
image_width = 224
batch_size = 32
base_learning_rate = 0.0001
epochs = 10
num_workers = os.cpu_count() // 2 if os.cpu_count() else 1 

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
# Augmentation
train_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(image_height, image_width), 
                         scale=(0.8, 1.0), ratio=(0.9, 1.1), antialias=True),
    v2.RandomRotation(degrees=15, interpolation=v2.InterpolationMode.BILINEAR, fill=0), 
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((image_height, image_width), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
full_dataset = datasets.ImageFolder(root=dataset_dir)
class_names = full_dataset.classes
number_of_classes = len(class_names)
print(f"Classes: {class_names}")

train_dataset_full = datasets.ImageFolder(root=dataset_dir, transform=train_transforms)
val_test_dataset_full = datasets.ImageFolder(root=dataset_dir, transform=val_test_transforms)


total_len = len(train_dataset_full)
val_len = int(0.15 * total_len)
test_len = int(0.15 * total_len)
train_len = total_len - val_len - test_len

print(f"Split: Train={train_len}, Val={val_len}, Test={test_len}")


indices = torch.randperm(total_len).tolist()

train_indices = indices[:train_len]
val_indices = indices[train_len : train_len + val_len]
test_indices = indices[train_len + val_len :]


train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_test_dataset_full, val_indices) 
test_dataset  = Subset(val_test_dataset_full, test_indices) 

# 6. Loaders
train_loader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers) 


Classes: ['Color Fundus', 'Diabetic Retinopathy', 'Disc Edema', 'Glaucoma', 'Healthy', 'Macular Scar', 'Myopia', 'Pterygium', 'Retinal Detachment', 'Retinitis Pigmentosa']
Split: Train=3735, Val=800, Test=800


In [5]:
denseNet121 = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
for param in denseNet121.features.parameters():
    param.requires_grad = False

num_filters = denseNet121.classifier.in_features
denseNet121.classifier = nn.Linear(num_filters, number_of_classes)
denseNet121 = denseNet121.to(device)
# print("Model details")
# print(denseNet121)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(denseNet121.parameters(), lr=base_learning_rate)


train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the denseNet121
    denseNet121.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = denseNet121(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    denseNet121.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = denseNet121(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch 1/10


Epoch 1 validation: 100%|██████████| 25/25 [00:32<00:00,  1.31s/it]


Train Loss: 1.8726, Train Acc: 0.2980
Val Loss: 1.7534, Val Acc: 0.3925

Epoch 2/10


Epoch 2 validation: 100%|██████████| 25/25 [00:28<00:00,  1.16s/it]


Train Loss: 1.6263, Train Acc: 0.4412
Val Loss: 1.5966, Val Acc: 0.4375

Epoch 3/10


Epoch 3 validation: 100%|██████████| 25/25 [00:27<00:00,  1.12s/it]


... (11 outputs left out)

Epoch 9 validation: 100%|██████████| 25/25 [00:27<00:00,  1.10s/it]


Train Loss: 1.1623, Train Acc: 0.6056
Val Loss: 1.1627, Val Acc: 0.5938

Epoch 10/10


Epoch 10 validation: 100%|██████████| 25/25 [00:27<00:00,  1.12s/it]

Train Loss: 1.1422, Train Acc: 0.6102
Val Loss: 1.1510, Val Acc: 0.5750


In [ ]:
def run_evaluation_metrics(y_true, y_preds, y_probs, class_names, model):

    num_classes = len(class_names)
    
    # 1. Calculate Metrics
    accuracy = accuracy_score(y_true, y_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_preds, average=None)
    macro_f1 = f1.mean()
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(y_true, y_preds, average='weighted')
    
    
    

    # AUC Scores
    try:
        auc_ovr = roc_auc_score(y_true, y_probs, multi_class='ovr', average='weighted')
        auc_macro = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        auc_weighted = roc_auc_score(y_true, y_probs, multi_class='ovr', average='weighted')
    except Exception as e:
        print(f"Could not calculate AUC: {e}")
        auc_ovr, auc_macro, auc_weighted = 0, 0, 0

    # Print Text Report
    print(f"\n{'='*10} RESULTS FOR: {model} {'='*10}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("\nPer-Class Performance:")
    for i, name in enumerate(class_names):
        print(f"  {name:15s}  Precision: {precision[i]:.4f}  Recall: {recall[i]:.4f}  F1: {f1[i]:.4f}")
    print(f"\nMacro F1-score: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Weighted AUC: {auc_ovr:.4f}")
    print(f"Macro AUC (OvR): {auc_macro:.4f}")
    print(f"Weighted AUC (OvR): {auc_weighted:.4f}")
    
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_preds, target_names=class_names))

    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model}: Confusion Matrix (Acc: {accuracy:.4f})')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Plot ROC Curves
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    
    plt.figure(figsize=(12, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, num_classes))

    for i, color in zip(range(num_classes), colors):
        if np.sum(y_true == i) > 0: 
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=color, lw=2, 
                     label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Chance')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model}: ROC Curves')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

In [6]:
efficientNetB0 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
for param in efficientNetB0.features.parameters():
    param.requires_grad = False


num_filters = efficientNetB0.classifier[1].in_features
efficientNetB0.classifier[1] = nn.Linear(num_filters, number_of_classes)
efficientNetB0 = efficientNetB0.to(device)
# print("Model details")
# print(efficientNetB0)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientNetB0.parameters(), lr=base_learning_rate)


train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    #training the efficientNetB0
    efficientNetB0.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = efficientNetB0(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)


    train_acc = correct / total
    train_loss = running_loss / total
    train_acc_list.append(train_acc)
    train_loss_list.append(train_loss)

    #evaluation
    efficientNetB0.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = efficientNetB0(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)



        val_acc = val_correct / val_total
        val_loss /= val_total
        val_acc_list.append(val_acc)
        val_loss_list.append(val_loss)


        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch 1/10


Epoch 1 validation: 100%|██████████| 25/25 [00:26<00:00,  1.08s/it]


Train Loss: 1.9265, Train Acc: 0.3331
Val Loss: 1.7308, Val Acc: 0.4587

Epoch 2/10


Epoch 2 validation: 100%|██████████| 25/25 [00:27<00:00,  1.09s/it]


Train Loss: 1.6357, Train Acc: 0.4784
Val Loss: 1.5628, Val Acc: 0.5012

Epoch 3/10


Epoch 3 validation: 100%|██████████| 25/25 [00:27<00:00,  1.10s/it]


... (11 outputs left out)

Epoch 9 validation: 100%|██████████| 25/25 [00:27<00:00,  1.12s/it]


Train Loss: 1.2300, Train Acc: 0.5936
Val Loss: 1.1887, Val Acc: 0.6100

Epoch 10/10


Epoch 10 validation: 100%|██████████| 25/25 [00:26<00:00,  1.08s/it]

Train Loss: 1.2023, Train Acc: 0.5885
Val Loss: 1.1616, Val Acc: 0.6175


In [5]:
googlenet_model = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
for param in googlenet_model.parameters():
    param.requires_grad = False

num_features = googlenet_model.fc.in_features
googlenet_model.fc = nn.Linear(num_features, number_of_classes) 
googlenet_model = googlenet_model.to(device)
print(f"Model {type(googlenet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(googlenet_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    googlenet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = googlenet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    googlenet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = googlenet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")




Model GoogLeNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:40<00:00,  1.63s/it]


Train Loss: 1.8732, Train Acc: 0.2718
Val Loss: 1.7582, Val Acc: 0.3912

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:40<00:00,  1.61s/it]


Train Loss: 1.6816, Train Acc: 0.4238
Val Loss: 1.6176, Val Acc: 0.4700

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:41<00:00,  1.66s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:34<00:00,  1.36s/it]


Train Loss: 1.2796, Train Acc: 0.5735
Val Loss: 1.2494, Val Acc: 0.5587

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]

Train Loss: 1.2562, Train Acc: 0.5681
Val Loss: 1.2160, Val Acc: 0.5787


In [6]:
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for param in resnet50.parameters():
    param.requires_grad = False

num_features = resnet50.fc.in_features
resnet50.fc = nn.Linear(num_features, number_of_classes) 
resnet50 = resnet50.to(device)
print(f"Model {type(resnet50).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet50.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    resnet50.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet50(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    resnet50.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet50(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model ResNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:40<00:00,  1.63s/it]


Train Loss: 1.9050, Train Acc: 0.3218
Val Loss: 1.7611, Val Acc: 0.3937

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:38<00:00,  1.53s/it]


Train Loss: 1.6710, Train Acc: 0.4578
Val Loss: 1.6488, Val Acc: 0.4675

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:39<00:00,  1.57s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.42s/it]


Train Loss: 1.2703, Train Acc: 0.5815
Val Loss: 1.3112, Val Acc: 0.5625

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.49s/it]

Train Loss: 1.2383, Train Acc: 0.5912
Val Loss: 1.2693, Val Acc: 0.5637


In [7]:
vgg16_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
for param in vgg16_model.features.parameters():
    param.requires_grad = False

num_features = vgg16_model.classifier[6].in_features
vgg16_model.classifier[6] = nn.Linear(num_features, number_of_classes) 
vgg16_model = vgg16_model.to(device)
print(f"Model {type(vgg16_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    vgg16_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg16_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    vgg16_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = vgg16_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model VGG initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.3666, Train Acc: 0.5207
Val Loss: 1.2515, Val Acc: 0.5062

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.1287, Train Acc: 0.5960
Val Loss: 1.0590, Val Acc: 0.6000

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 0.7754, Train Acc: 0.7133
Val Loss: 0.8802, Val Acc: 0.6850

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]

Train Loss: 0.7603, Train Acc: 0.7162
Val Loss: 0.9001, Val Acc: 0.6687


In [8]:
vgg19_model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
for param in vgg19_model.features.parameters():
    param.requires_grad = False

num_features = vgg19_model.classifier[6].in_features
vgg19_model.classifier[6] = nn.Linear(num_features, number_of_classes) 
vgg19_model = vgg19_model.to(device)
print(f"Model {type(vgg19_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg19_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    vgg19_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg19_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    vgg19_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = vgg19_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model VGG initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.44s/it]


Train Loss: 1.4010, Train Acc: 0.5138
Val Loss: 1.2940, Val Acc: 0.5525

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.43s/it]


Train Loss: 1.1580, Train Acc: 0.5780
Val Loss: 1.2247, Val Acc: 0.5363

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.43s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.43s/it]


Train Loss: 0.8569, Train Acc: 0.6760
Val Loss: 1.0135, Val Acc: 0.6312

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.41s/it]

Train Loss: 0.8204, Train Acc: 0.6934
Val Loss: 0.9786, Val Acc: 0.6212


In [9]:
mobilenet_model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
for param in mobilenet_model.features.parameters():
    param.requires_grad = False

num_features = mobilenet_model.classifier[1].in_features
mobilenet_model.classifier[1] = nn.Linear(num_features, number_of_classes) 
mobilenet_model = mobilenet_model.to(device)
print(f"Model {type(mobilenet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    mobilenet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = mobilenet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    mobilenet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = mobilenet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")




Model MobileNetV2 initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.43s/it]


Train Loss: 1.7542, Train Acc: 0.3561
Val Loss: 1.5942, Val Acc: 0.4562

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.42s/it]


Train Loss: 1.4936, Train Acc: 0.5151
Val Loss: 1.4074, Val Acc: 0.5500

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.42s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.41s/it]


Train Loss: 1.1179, Train Acc: 0.6123
Val Loss: 1.0570, Val Acc: 0.6288

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.43s/it]

Train Loss: 1.0782, Train Acc: 0.6286
Val Loss: 1.0594, Val Acc: 0.6312


In [11]:
alexNet_model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

for param in alexNet_model.parameters():
    param.requires_grad = False

num_features = alexNet_model.classifier[6].in_features
alexNet_model.classifier[6] = nn.Linear(num_features, number_of_classes) 


alexNet_model = alexNet_model.to(device)
print(f"Model {type(alexNet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexNet_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    alexNet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = alexNet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    alexNet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = alexNet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model AlexNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:41<00:00,  1.67s/it]


Train Loss: 1.5499, Train Acc: 0.4613
Val Loss: 1.3242, Val Acc: 0.5275

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:39<00:00,  1.57s/it]


Train Loss: 1.2404, Train Acc: 0.5649
Val Loss: 1.1899, Val Acc: 0.5850

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:39<00:00,  1.59s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:38<00:00,  1.54s/it]


Train Loss: 1.0105, Train Acc: 0.6321
Val Loss: 1.0098, Val Acc: 0.6412

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.48s/it]

Train Loss: 0.9895, Train Acc: 0.6343
Val Loss: 1.0094, Val Acc: 0.6362


In [5]:
xception_model = timm.create_model('xception', pretrained=True)


for param in xception_model.parameters():
    param.requires_grad = False

num_features = xception_model.fc.in_features
xception_model.fc = nn.Linear(num_features, number_of_classes) 



xception_model = xception_model.to(device)
print(f"Model {type(xception_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(xception_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    xception_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = xception_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    xception_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = xception_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



d:\Practise\venv\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Model Xception initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:40<00:00,  1.62s/it]


Train Loss: 1.8872, Train Acc: 0.3673
Val Loss: 1.7365, Val Acc: 0.4338

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:38<00:00,  1.55s/it]


Train Loss: 1.6920, Train Acc: 0.4782
Val Loss: 1.6324, Val Acc: 0.5112

Epoch 3/10


... (12 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.3057, Train Acc: 0.5815
Val Loss: 1.2944, Val Acc: 0.5725

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]

Train Loss: 1.2896, Train Acc: 0.5788
Val Loss: 1.2780, Val Acc: 0.5875


In [8]:
model_name = 'inception_resnet_v2'
inception_resnet_v2_model = timm.create_model(model_name, pretrained=True,  num_classes=number_of_classes)


for param in inception_resnet_v2_model.parameters():
    param.requires_grad = False

number_of_features = inception_resnet_v2_model.classif.in_features
inception_resnet_v2_model.classif = nn.Linear(number_of_features, number_of_classes)

inception_resnet_v2_model = inception_resnet_v2_model.to(device)

print(f"Model {model_name} initialized for {number_of_classes} global classes.")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, inception_resnet_v2_model.parameters()), lr=base_learning_rate)


train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    # Training 
    inception_resnet_v2_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = inception_resnet_v2_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    inception_resnet_v2_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = inception_resnet_v2_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Model inception_resnet_v2 initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.8495, Train Acc: 0.3730
Val Loss: 1.7378, Val Acc: 0.4462

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.6915, Train Acc: 0.4774
Val Loss: 1.6321, Val Acc: 0.4763

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.3722, Train Acc: 0.5510
Val Loss: 1.3360, Val Acc: 0.5550

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:35<00:00,  1.44s/it]

Train Loss: 1.3487, Train Acc: 0.5598
Val Loss: 1.3193, Val Acc: 0.5563


In [9]:
shuffleNet_model = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)


for param in shuffleNet_model.parameters():
    param.requires_grad = False

num_features = shuffleNet_model.fc.in_features
shuffleNet_model.fc = nn.Linear(num_features, number_of_classes) 



shuffleNet_model = shuffleNet_model.to(device)
print(f"Model {type(shuffleNet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(shuffleNet_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    shuffleNet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = shuffleNet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    shuffleNet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = shuffleNet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model ShuffleNetV2 initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


Train Loss: 2.2375, Train Acc: 0.2522
Val Loss: 2.1673, Val Acc: 0.3812

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.51s/it]


Train Loss: 2.1170, Train Acc: 0.3448
Val Loss: 2.0627, Val Acc: 0.3113

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


Train Loss: 1.7890, Train Acc: 0.4324
Val Loss: 1.7706, Val Acc: 0.4387

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.49s/it]

Train Loss: 1.7718, Train Acc: 0.4356
Val Loss: 1.7505, Val Acc: 0.4412


In [10]:
squeezeNet_model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.IMAGENET1K_V1)


for param in squeezeNet_model.parameters():
    param.requires_grad = False

original_conv_layer = squeezeNet_model.classifier[1]
num_input_features = original_conv_layer.in_channels 

squeezeNet_model.classifier[1] = nn.Conv2d(
    in_channels=num_input_features,
    out_channels=number_of_classes,
    kernel_size=(1, 1), 
    stride=(1, 1)
)


for param in squeezeNet_model.classifier[1].parameters():
    param.requires_grad = True



squeezeNet_model = squeezeNet_model.to(device)
print(f"Model {type(squeezeNet_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(squeezeNet_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    squeezeNet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = squeezeNet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    squeezeNet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
            images, labels = images.to(device), labels.to(device)
            outputs = squeezeNet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model SqueezeNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.49s/it]


Train Loss: 1.9250, Train Acc: 0.3301
Val Loss: 1.6731, Val Acc: 0.3937

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


Train Loss: 1.5953, Train Acc: 0.4402
Val Loss: 1.4751, Val Acc: 0.5000

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.49s/it]


Train Loss: 1.1097, Train Acc: 0.6112
Val Loss: 1.0983, Val Acc: 0.6225

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.48s/it]

Train Loss: 1.0802, Train Acc: 0.6187
Val Loss: 1.0421, Val Acc: 0.6550


In [11]:
resnet101_model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
for param in resnet101_model.parameters():
    param.requires_grad = False

num_features = resnet101_model.fc.in_features
resnet101_model.fc = nn.Linear(num_features, number_of_classes) 
resnet101_model = resnet101_model.to(device)
print(f"Model {type(resnet101_model).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet101_model.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    resnet101_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet101_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    resnet101_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet101_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model ResNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


Train Loss: 1.8441, Train Acc: 0.3649
Val Loss: 1.6844, Val Acc: 0.4725

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.6295, Train Acc: 0.4693
Val Loss: 1.5587, Val Acc: 0.5050

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.2475, Train Acc: 0.5778
Val Loss: 1.2330, Val Acc: 0.5938

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]

Train Loss: 1.2281, Train Acc: 0.5823
Val Loss: 1.1981, Val Acc: 0.6025


In [12]:
darkNetmodel = timm.create_model('cspdarknet53.ra_in1k', pretrained=True)

for param in darkNetmodel.parameters():
    param.requires_grad = False

num_features = darkNetmodel.head.fc.in_features
darkNetmodel.head.fc = nn.Linear(num_features, number_of_classes) 
darkNetmodel = darkNetmodel.to(device)
print(f"Model {type(darkNetmodel).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(darkNetmodel.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    darkNetmodel.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = darkNetmodel(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    darkNetmodel.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = darkNetmodel(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model CspNet initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.9934, Train Acc: 0.3151
Val Loss: 1.7722, Val Acc: 0.4275

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


Train Loss: 1.7343, Train Acc: 0.4458
Val Loss: 1.6390, Val Acc: 0.4700

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.3286, Train Acc: 0.5695
Val Loss: 1.3010, Val Acc: 0.5663

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]

Train Loss: 1.3144, Train Acc: 0.5620
Val Loss: 1.2834, Val Acc: 0.5663


In [13]:
vision_transformer = timm.create_model('vit_base_patch16_224', pretrained=True)


for param in vision_transformer.parameters():
    param.requires_grad = False

num_features = vision_transformer.head.in_features
vision_transformer.head = nn.Linear(num_features, number_of_classes) 
vision_transformer = vision_transformer.to(device)
print(f"Model {type(vision_transformer).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vision_transformer.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    vision_transformer.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vision_transformer(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    vision_transformer.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = vision_transformer(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model VisionTransformer initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 1.7556, Train Acc: 0.3751
Val Loss: 1.4782, Val Acc: 0.4975

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


Train Loss: 1.3828, Train Acc: 0.5451
Val Loss: 1.2842, Val Acc: 0.5750

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Train Loss: 0.9429, Train Acc: 0.6774
Val Loss: 0.9473, Val Acc: 0.6650

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]

Train Loss: 0.9279, Train Acc: 0.6806
Val Loss: 0.9308, Val Acc: 0.6700


In [14]:
max_vit = timm.create_model('maxvit_tiny_tf_224.in1k', pretrained=True, num_classes=number_of_classes)


for param in max_vit.parameters():
    param.requires_grad = False

for param in max_vit.head.parameters():
    param.requires_grad = True

    
max_vit = max_vit.to(device)
print(f"Model {type(max_vit).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(max_vit.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    max_vit.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = max_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    max_vit.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = max_vit(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model MaxxVit initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.50s/it]


Train Loss: 1.7458, Train Acc: 0.4179
Val Loss: 1.5731, Val Acc: 0.4300

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.48s/it]


Train Loss: 1.4244, Train Acc: 0.5272
Val Loss: 1.4013, Val Acc: 0.5175

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:37<00:00,  1.48s/it]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Train Loss: 1.0259, Train Acc: 0.6343
Val Loss: 1.0664, Val Acc: 0.6350

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:38<00:00,  1.52s/it]

Train Loss: 0.9960, Train Acc: 0.6511
Val Loss: 1.0557, Val Acc: 0.6238


In [5]:
pvt_vit = timm.create_model('pvt_v2_b0', pretrained=True, num_classes=number_of_classes)


for param in pvt_vit.parameters():
    param.requires_grad = False

for param in pvt_vit.head.parameters():
    param.requires_grad = True

num_features = pvt_vit.head.in_features
pvt_vit.head = nn.Linear(num_features, number_of_classes) 

pvt_vit = pvt_vit.to(device)
print(f"Model {type(pvt_vit).__name__} initialized for {number_of_classes} global classes.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pvt_vit.parameters(), lr=base_learning_rate)



train_acc_list, val_acc_list = [], []
train_loss_list, val_loss_list = [], []

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    pvt_vit.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"): 
    # for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = pvt_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    pvt_vit.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"): 
        # for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = pvt_vit(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Model PyramidVisionTransformerV2 initialized for 10 global classes.

Epoch 1/10


Epoch 1 Validation: 100%|██████████| 25/25 [00:22<00:00,  1.13it/s]


Train Loss: 2.0285, Train Acc: 0.2490
Val Loss: 1.8703, Val Acc: 0.3050

Epoch 2/10


Epoch 2 Validation: 100%|██████████| 25/25 [00:20<00:00,  1.21it/s]


Train Loss: 1.6855, Train Acc: 0.3989
Val Loss: 1.6751, Val Acc: 0.4125

Epoch 3/10


Epoch 3 Validation: 100%|██████████| 25/25 [00:20<00:00,  1.25it/s]


... (11 outputs left out)

Epoch 9 Validation: 100%|██████████| 25/25 [00:20<00:00,  1.23it/s]


Train Loss: 1.1808, Train Acc: 0.5944
Val Loss: 1.2220, Val Acc: 0.5700

Epoch 10/10


Epoch 10 Validation: 100%|██████████| 25/25 [00:20<00:00,  1.20it/s]

Train Loss: 1.1373, Train Acc: 0.6064
Val Loss: 1.1910, Val Acc: 0.5725
